# SRGCD: Stability-Driven Region Growth for 3D Change Detection
Colab-ready PyTorch implementation. Modules: KPConv-FPN · MGCP · SGCA (SAP+SPT) · BCE+CBL Loss · mIoU/Dice Eval.

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install open3d scikit-learn scipy tqdm matplotlib -q
import os, sys
if not os.path.exists('KPConv'):
    os.system('git clone https://github.com/HuguesTHOMAS/KPConv.git')
sys.path.insert(0, 'KPConv')
print('Setup complete.')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from scipy.spatial import KDTree
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
def estimate_normals_pca(points, k=20):
    tree = KDTree(points)
    normals = np.zeros_like(points)
    for i, p in enumerate(points):
        _, idx = tree.query(p, k=k)
        nb = points[idx]
        centred = nb - nb.mean(0)
        cov = centred.T @ centred
        _, vecs = np.linalg.eigh(cov)
        normals[i] = vecs[:, 0]
    flip = (normals * (-points)).sum(-1) < 0
    normals[flip] *= -1
    return normals

def nn_difference_features(pts_t1, feats_t1, pts_t2, feats_t2, k=1):
    tree = KDTree(pts_t2)
    _, idx = tree.query(pts_t1, k=k)
    if k == 1:
        idx = idx.reshape(-1, 1)
    nn_feats = feats_t2[idx].mean(1)
    return feats_t1 - nn_feats

def preprocess_point_cloud(pts, rgb=None, k_normal=20):
    normals = estimate_normals_pca(pts, k=k_normal)
    parts = [pts, normals]
    if rgb is not None:
        parts.append(rgb)
    return np.concatenate(parts, axis=-1).astype(np.float32)

print('Preprocessing utilities ready.')

In [ ]:
class ChangeDetectionDataset(Dataset):
    """
    Synthetic placeholder dataset.
    Replace __getitem__ with your own file I/O:
      e.g. load .ply / .npy / .las files from Google Drive.
    Each sample returns:
      pts_t1  (N,3)  point coords at time T1
      pts_t2  (N,3)  point coords at time T2
      feats   (N,18) [f1_6 | f2_6 | diff_6]
      labels  (N,)   binary int64 {0=unchanged, 1=changed}
    """
    def __init__(self, num_samples=64, num_points=512, seed=42):
        super().__init__()
        rng = np.random.default_rng(seed)
        self.samples = []
        for _ in range(num_samples):
            pts1 = rng.uniform(-1, 1, (num_points, 3)).astype(np.float32)
            pts2 = pts1 + rng.normal(0, 0.02, pts1.shape).astype(np.float32)
            lbl  = (rng.random(num_points) > 0.8).astype(np.int64)
            self.samples.append((pts1, pts2, lbl))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pts1, pts2, labels = self.samples[idx]
        f1   = preprocess_point_cloud(pts1)
        f2   = preprocess_point_cloud(pts2)
        diff = nn_difference_features(pts1, f1, pts2, f2, k=1)
        feat = np.concatenate([f1, f2, diff], axis=-1)
        return {
            'pts_t1' : torch.from_numpy(pts1),
            'pts_t2' : torch.from_numpy(pts2),
            'feats'  : torch.from_numpy(feat),
            'labels' : torch.from_numpy(labels),
        }

_d = ChangeDetectionDataset(num_samples=2, num_points=64)
_s = _d[0]
print('feats shape :', _s['feats'].shape)
print('labels shape:', _s['labels'].shape)
print('Dataset OK.')

In [ ]:
# KPConvLayer approximates the KPConv operation as a shared-weight MLP.
# To use the real KPConv CUDA kernel, import from the cloned KPConv repo:
#   from KPConv.models.architectures import KPFCNN

class KPConvLayer(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_ch, out_ch),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        B, N, C = x.shape
        return self.mlp(x.reshape(B * N, C)).reshape(B, N, -1)

class FPNFusion(nn.Module):
    def __init__(self, ch_deep, ch_shallow, out_ch):
        super().__init__()
        self.top = nn.Linear(ch_deep,    out_ch)
        self.lat = nn.Linear(ch_shallow, out_ch)
        self.out = nn.Linear(out_ch,     out_ch)
    def forward(self, deep, shallow):
        return F.relu(self.out(F.relu(self.top(deep)) + F.relu(self.lat(shallow))))

class KPConvFPN(nn.Module):
    """
    Siamese dual-branch KPConv-FPN encoder.
    Both branches share weights (one set of parameters).
    3 encoder stages: in_ch -> 64 -> 128 -> 256
    2 FPN top-down fusions -> out_ch=128 per branch.
    """
    def __init__(self, in_ch=18, base_ch=64, out_ch=128):
        super().__init__()
        self.e1 = KPConvLayer(in_ch,     base_ch)
        self.e2 = KPConvLayer(base_ch,   base_ch * 2)
        self.e3 = KPConvLayer(base_ch*2, base_ch * 4)
        self.f1 = FPNFusion(base_ch * 4, base_ch * 2, base_ch * 2)
        self.f2 = FPNFusion(base_ch * 2, base_ch,     out_ch)

    def _branch(self, x):
        s1 = self.e1(x)
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        return self.f2(self.f1(s3, s2), s1)

    def forward(self, f_t1, f_t2):
        return self._branch(f_t1), self._branch(f_t2)

print('KPConv-FPN backbone ready.')

In [ ]:
class MGCPSeedInitializer(nn.Module):
    """
    Mutual Geometric Consistency Prior.
    Computes per-point stability score s in [0,1]:
      s = (normal_angle_diff + feature_cosine_dist + learned_score) / 3
    High s -> geometrically inconsistent -> 'changed' seed
    Low  s -> geometrically consistent   -> 'unchanged' seed
    """
    def __init__(self, feat_ch=128, thresh_high=0.7, thresh_low=0.3):
        super().__init__()
        self.thresh_high = thresh_high
        self.thresh_low  = thresh_low
        self.proj = nn.Sequential(
            nn.Linear(feat_ch * 2, feat_ch),
            nn.ReLU(inplace=True),
            nn.Linear(feat_ch, 1),
            nn.Sigmoid(),
        )

    def forward(self, enc_t1, enc_t2, normals_t1, normals_t2):
        n1 = F.normalize(normals_t1, dim=-1)
        n2 = F.normalize(normals_t2, dim=-1)
        norm_diff = (1.0 - (n1 * n2).sum(-1, keepdim=True)) / 2.0

        f1 = F.normalize(enc_t1, dim=-1)
        f2 = F.normalize(enc_t2, dim=-1)
        feat_diff = (1.0 - (f1 * f2).sum(-1, keepdim=True)) / 2.0

        learned = self.proj(torch.cat([enc_t1, enc_t2], dim=-1))
        score   = (norm_diff + feat_diff + learned) / 3.0

        s = score.squeeze(-1)
        return score, s >= self.thresh_high, s <= self.thresh_low

print('MGCP ready.')

In [ ]:
class StabilityAdaptivePropagation(nn.Module):
    """SAP: propagates features from stable seeds to neighbours."""
    def __init__(self, feat_ch=128, k=16):
        super().__init__()
        self.k    = k
        self.gate = nn.Sequential(nn.Linear(feat_ch + 1, feat_ch), nn.Sigmoid())
        self.proj = nn.Linear(feat_ch, feat_ch)

    def forward(self, feats, stability):
        B, N, C = feats.shape
        with torch.no_grad():
            sim  = torch.bmm(feats, feats.transpose(1, 2))
            topk = sim.topk(self.k + 1, dim=-1).indices[:, :, 1:]

        idx_e    = topk.unsqueeze(-1).expand(-1, -1, -1, C)
        nb_feats = feats.unsqueeze(1).expand(-1, N, -1, -1)
        nb_feats = torch.gather(nb_feats, 2, idx_e)

        idx_s   = topk.unsqueeze(-1)
        nb_stab = torch.gather(stability.unsqueeze(1).expand(-1, N, -1, -1), 2, idx_s)

        w   = F.softmax(nb_stab, dim=2)
        agg = (w * nb_feats).sum(2)

        g = self.gate(torch.cat([agg, stability], dim=-1))
        return self.proj(g * agg + (1 - g) * feats)


class StabilityProportionalTemperature(nn.Module):
    """SPT: per-point adaptive softmax temperature for cross-attention."""
    def __init__(self, tau_base=1.0, alpha=2.0):
        super().__init__()
        self.tau_base = tau_base
        self.alpha    = alpha

    def forward(self, query, key, value, stability):
        C      = query.size(-1)
        tau    = self.tau_base * (1.0 + self.alpha * (1.0 - stability))
        scores = torch.bmm(query, key.transpose(1, 2)) / (C ** 0.5) / tau
        return torch.bmm(F.softmax(scores, dim=-1), value)


class SGCAModule(nn.Module):
    """SGCA: SAP -> cross-branch SPT attention -> residual + LayerNorm."""
    def __init__(self, feat_ch=128, k_sap=16):
        super().__init__()
        self.sap    = StabilityAdaptivePropagation(feat_ch, k=k_sap)
        self.spt    = StabilityProportionalTemperature()
        self.q_proj = nn.Linear(feat_ch, feat_ch)
        self.k_proj = nn.Linear(feat_ch, feat_ch)
        self.v_proj = nn.Linear(feat_ch, feat_ch)
        self.norm1  = nn.LayerNorm(feat_ch)
        self.norm2  = nn.LayerNorm(feat_ch)
        self.ffn    = nn.Sequential(
            nn.Linear(feat_ch, feat_ch * 2),
            nn.GELU(),
            nn.Linear(feat_ch * 2, feat_ch),
        )

    def forward(self, feat_t1, feat_t2, stability):
        p1  = self.sap(feat_t1, stability)
        p2  = self.sap(feat_t2, 1.0 - stability)
        Q   = self.q_proj(p1)
        K   = self.k_proj(p2)
        V   = self.v_proj(p2)
        x   = self.norm1(p1 + self.spt(Q, K, V, stability))
        return self.norm2(x + self.ffn(x))

print('SGCA (SAP + SPT) ready.')

In [ ]:
class SRGCD(nn.Module):
    """
    Full SRGCD pipeline:
      KPConv-FPN -> MGCP seed init -> SGCA x n_sgca -> MLP head
    """
    def __init__(self, in_ch=18, base_ch=64, enc_out=128,
                 n_sgca=2, thresh_high=0.7, thresh_low=0.3):
        super().__init__()
        self.backbone = KPConvFPN(in_ch=in_ch, base_ch=base_ch, out_ch=enc_out)
        self.mgcp     = MGCPSeedInitializer(enc_out, thresh_high, thresh_low)
        self.sgca     = nn.ModuleList([SGCAModule(enc_out) for _ in range(n_sgca)])
        self.head     = nn.Sequential(
            nn.Linear(enc_out, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, batch):
        feats = batch['feats'].to(DEVICE)
        f1    = feats[:, :, :6]
        f2    = feats[:, :, 6:12]
        n1    = f1[:, :, 3:6]
        n2    = f2[:, :, 3:6]

        e1, e2               = self.backbone(f1, f2)
        stab, seed_ch, seed_unch = self.mgcp(e1, e2, n1, n2)

        x = e1
        for layer in self.sgca:
            x = layer(x, e2, stab)

        logits = self.head(x).squeeze(-1)
        return {
            'logits'         : logits,
            'stability'      : stab.squeeze(-1),
            'sgca_feats'     : x,
            'seed_changed'   : seed_ch,
            'seed_unchanged' : seed_unch,
        }

# --- Shape test ---
_model  = SRGCD().to(DEVICE)
_ds     = ChangeDetectionDataset(num_samples=2, num_points=128)
_dl     = DataLoader(_ds, batch_size=2)
_b      = {k: v.to(DEVICE) for k, v in next(iter(_dl)).items()}
_o      = _model(_b)
print('logits    :', _o['logits'].shape)
print('stability :', _o['stability'].shape)
params  = sum(p.numel() for p in _model.parameters() if p.requires_grad)
print(f'Trainable params: {params:,}')
print('SRGCD model OK.')

In [ ]:
class BCELoss(nn.Module):
    def __init__(self, pos_weight=4.0):
        super().__init__()
        self.fn = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(pos_weight).to(DEVICE))
    def forward(self, logits, labels):
        return self.fn(logits, labels.float())


class ContrastiveBoundaryLoss(nn.Module):
    """
    CBL: InfoNCE on boundary-proximal points (stability in (0.3, 0.7)).
    Pulls same-class embeddings together, pushes different-class apart.
    """
    def __init__(self, feat_ch=128, proj_ch=64, temperature=0.07):
        super().__init__()
        self.T    = temperature
        self.proj = nn.Sequential(
            nn.Linear(feat_ch, proj_ch),
            nn.ReLU(),
            nn.Linear(proj_ch, proj_ch),
        )

    def forward(self, feats, labels, stability):
        B, N, C = feats.shape
        z        = F.normalize(self.proj(feats), dim=-1)
        boundary = (stability > 0.3) & (stability < 0.7)
        total, count = torch.tensor(0.0, device=feats.device), 0
        for b in range(B):
            idx = boundary[b].nonzero(as_tuple=True)[0]
            if len(idx) < 2:
                continue
            if len(idx) > 256:
                idx = idx[torch.randperm(len(idx))[:256]]
            zb  = z[b][idx]
            lb  = labels[b][idx]
            sim = zb @ zb.T / self.T
            pos = (lb.unsqueeze(0) == lb.unsqueeze(1)).float()
            pos.fill_diagonal_(0)
            exp   = sim.exp()
            denom = (exp.sum(1) - exp.diag()).clamp(min=1e-8)
            numer = (exp * pos).sum(1).clamp(min=1e-8)
            total = total + (-torch.log(numer / denom)).mean()
            count += 1
        return total / max(count, 1)


class SRGCDLoss(nn.Module):
    def __init__(self, feat_ch=128, lam_bce=1.0, lam_cbl=0.5):
        super().__init__()
        self.bce     = BCELoss()
        self.cbl     = ContrastiveBoundaryLoss(feat_ch=feat_ch)
        self.lam_bce = lam_bce
        self.lam_cbl = lam_cbl

    def forward(self, out, batch):
        labels = batch['labels'].to(DEVICE)
        l_bce  = self.bce(out['logits'], labels)
        l_cbl  = self.cbl(out['sgca_feats'], labels, out['stability'])
        total  = self.lam_bce * l_bce + self.lam_cbl * l_cbl
        return {'total': total, 'bce': l_bce, 'cbl': l_cbl}

print('BCE + CBL loss functions ready.')

In [ ]:
def compute_miou(preds, labels):
    eps = 1e-6
    ious = []
    for c in [0, 1]:
        tp = ((preds == c) & (labels == c)).sum()
        fp = ((preds == c) & (labels != c)).sum()
        fn = ((preds != c) & (labels == c)).sum()
        ious.append(tp / (tp + fp + fn + eps))
    return float(np.mean(ious))


def compute_dice(preds, labels):
    eps = 1e-6
    tp  = ((preds == 1) & (labels == 1)).sum()
    fp  = ((preds == 1) & (labels == 0)).sum()
    fn  = ((preds == 0) & (labels == 1)).sum()
    return float(2 * tp / (2 * tp + fp + fn + eps))


def evaluate_model(model, loader):
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating'):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out   = model(batch)
            preds = (torch.sigmoid(out['logits']) > 0.5).long()
            all_p.append(preds.cpu().reshape(-1))
            all_l.append(batch['labels'].cpu().reshape(-1))
    p    = torch.cat(all_p).numpy()
    l    = torch.cat(all_l).numpy()
    miou = compute_miou(p, l)
    dice = compute_dice(p, l)
    for c, name in [(0, 'Unchanged'), (1, 'Changed')]:
        mask = l == c
        acc  = (p[mask] == l[mask]).mean() if mask.sum() > 0 else 0.0
        print(f'  {name:12s}: Acc={acc:.4f}')
    print(f'  mIoU = {miou:.4f}')
    print(f'  Dice = {dice:.4f}')
    return {'miou': miou, 'dice': dice}

print('Evaluation utilities ready.')

In [ ]:
def train_srgcd(num_epochs=20, batch_size=4, lr=1e-3, num_points=512):
    train_ds = ChangeDetectionDataset(num_samples=128, num_points=num_points, seed=0)
    val_ds   = ChangeDetectionDataset(num_samples=32,  num_points=num_points, seed=99)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size)

    model     = SRGCD().to(DEVICE)
    criterion = SRGCDLoss().to(DEVICE)
    opt       = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched     = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs)

    best_miou = 0.0
    history   = {'train_loss': [], 'val_miou': [], 'val_dice': []}

    for epoch in range(1, num_epochs + 1):
        model.train()
        ep_loss = 0.0
        for batch in tqdm(train_dl, desc=f'Ep {epoch}/{num_epochs}', leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            opt.zero_grad()
            out    = model(batch)
            losses = criterion(out, batch)
            losses['total'].backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            ep_loss += losses['total'].item()
        sched.step()
        avg_loss = ep_loss / len(train_dl)
        history['train_loss'].append(avg_loss)

        model.eval()
        all_p, all_l = [], []
        with torch.no_grad():
            for batch in val_dl:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                out   = model(batch)
                preds = (torch.sigmoid(out['logits']) > 0.5).long()
                all_p.append(preds.cpu().reshape(-1))
                all_l.append(batch['labels'].cpu().reshape(-1))
        p    = torch.cat(all_p).numpy()
        l    = torch.cat(all_l).numpy()
        miou = compute_miou(p, l)
        dice = compute_dice(p, l)
        history['val_miou'].append(miou)
        history['val_dice'].append(dice)

        if miou > best_miou:
            best_miou = miou
            torch.save(model.state_dict(), 'srgcd_best.pth')

        print(f'Epoch {epoch:3d} | Loss {avg_loss:.4f} | mIoU {miou:.4f} | Dice {dice:.4f}')

    print(f'Best val mIoU: {best_miou:.4f}')
    return model, history


# --- Run ---
trained_model, history = train_srgcd(num_epochs=20, batch_size=4, lr=1e-3, num_points=512)

# --- Final evaluation ---
print('\n=== Final Evaluation ===')
test_ds = ChangeDetectionDataset(num_samples=32, num_points=512, seed=777)
test_dl = DataLoader(test_ds, batch_size=4)
trained_model.load_state_dict(torch.load('srgcd_best.pth', map_location=DEVICE))
metrics = evaluate_model(trained_model, test_dl)

# --- Plot ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], color='steelblue', label='Train Loss')
ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['val_miou'], color='darkorange', label='Val mIoU')
ax2.plot(history['val_dice'], color='green',      label='Val Dice')
ax2.set_title('Validation Metrics'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout()
plt.savefig('srgcd_training.png', dpi=150)
plt.show()
print('Done. Plot saved to srgcd_training.png')